In [1]:
import argparse
import time
import numpy as np
import wandb

from reasoning.tools.utils import load_model_with_vllm
from datasets import load_dataset
from vllm import SamplingParams
from reasoning.inference.majority import MajorityInference
from reasoning.tools.logger import load_config, apply_config

from reasoning.evaluator.math_grader import math_equal, extract_answer

from reasoning.models.model import ValueModel_qwen


config = load_config("/ssdscratch/byuan48/efficient_reasoning/configs/experiments_418/development.yaml")
apply_config(config) 
print(config)

# load dataset and model
dataset = load_dataset(config["data_path"])
dataset = dataset['test']
model_name = config["model_name"]

policy_model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=len(config["cuda_device_ids"]), gpu_memory_utilization=0.8)
tokenizer.pad_token = tokenizer.eos_token 
reward_model = ValueModel_qwen(device = "auto") # default one
start_problem_index = config.get("begin_problem_index", 0)
end_problem_index = config.get("end_problem_index", len(dataset)-1)
dataset = dataset.select(range(start_problem_index, end_problem_index)) # need to check this

# inference hyperparameters
num_return_sequences = config["num_return_sequences"]
assert num_return_sequences == 1
max_new_tokens = config["max_new_tokens"]
temperature = config["temperature"]
top_p = config["top_p"]
system_prompt = config["system_prompt"]

# SamplingTree parameters
sampling_method = config["sampling_method"]
samplingTree_temperature = config["samplingTree_temperature"]
beam_width = config["beam_width"]
max_steps = config["max_steps"]
threshold = config["threshold"]
ORM_type = config["ORM_type"]
# name of the results file
config_name = config["config_name"]


# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
    seed  = config["seed"]
) # shouldn't set seed for random sampling


wandb: Currently logged in as: yuanbo096. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'config_name': 'test', 'cuda_device_ids': [0, 1, 2, 3], 'seed': 1000, 'data_path': 'HuggingFaceH4/MATH-500', 'model_name': 'meta-llama/Llama-3.2-1B-Instruct', 'reward_model_name': 'Qwen/Qwen2.5-Math-PRM-7B', 'batch_size': 1, 'max_new_tokens': 2048, 'temperature': 0.7, 'top_p': 0.9, 'sampling_method': 'stochastic_beam_search_version4', 'num_return_sequences': 1, 'samplingTree_temperature': 0.1, 'beam_width': 4, 'max_steps': 2, 'threshold': 0.9, 'wandb_project': 'efficient_reasoning', 'ORM_type': 'product', 'system_prompt': 'Solve the following math problem efficiently and clearly:\n\n- For simple problems (2 steps or fewer):\n  Provide a concise solution with minimal explanation.\n\n- For complex problems (3 steps or more):\n  Use this step-by-step format:\n\n  ## Step 1: [Concise description]\n  [Brief explanation and calculations]\n\n  ## Step 2: [Concise description]\n  [Brief explanation and calculations]\n\n  ...\n\n  Regardless of the approach, always conclude with:\n\n  Therefor

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorkerProcess pid=3971702) INFO 04-19 18:26:03 weight_utils.py:296] No model.safetensors.index.json found in remote.
(VllmWorkerProcess pid=3971701) INFO 04-19 18:26:03 weight_utils.py:296] No model.safetensors.index.json found in remote.
INFO 04-19 18:26:03 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=3971700) INFO 04-19 18:26:03 weight_utils.py:296] No model.safetensors.index.json found in remote.
(VllmWorkerProcess pid=3971702) INFO 04-19 18:26:03 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=3971701) INFO 04-19 18:26:03 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=3971700) INFO 04-19 18:26:03 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=3971702) INFO 04-19 18:26:07 worker.py:241] Memory profiling takes 3.14 seconds
(VllmWorkerProcess pid=3971702) INFO 04-19 18:26:07 worker.py:241] the current vLLM instance can use total_gpu_memory (

Capturing CUDA graph shapes:  97%|█████████▋| 34/35 [00:17<00:00,  1.88it/s]

(VllmWorkerProcess pid=3971700) INFO 04-19 18:26:30 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB
(VllmWorkerProcess pid=3971701) INFO 04-19 18:26:30 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB
(VllmWorkerProcess pid=3971702) INFO 04-19 18:26:30 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:19<00:00,  1.83it/s]

INFO 04-19 18:26:30 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB
INFO 04-19 18:26:30 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 26.52 seconds


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at Qwen/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [2]:
import importlib
import reasoning.inference.tree
tree_module = importlib.reload(reasoning.inference.tree)
Tree = tree_module.Tree

In [3]:
start_time = time.time()
right_count = 0
num_generated_tokens = 0
error_steps_count_all = 0
print(f'length of the dataset: {len(dataset)}')
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]        

    tree = Tree(system_prompt, question, policy_model, reward_model, sampling_method, config_name, sampling_params, samplingTree_temperature, beam_width, threshold, ORM_type)
    for _ in range(max_steps):
        path = tree.generate_next_trajectory() 
        tree.paths.append(path)
    majority_inference = MajorityInference(policy_model=policy_model, tokenizer=tokenizer, sampling_params=sampling_params, config_name=config_name, reward_model=reward_model, method = 'weighted_majority', ORM_type=ORM_type)
    all_solutions  = [path.solutions for path in tree.explored_paths]
    rewards = [tree.get_reward(path.scores) for path in tree.explored_paths]
    
    solution = majority_inference.weighted_majority_vote(all_solutions, rewards)   
    extracted_answer = extract_answer(answer)
    is_correct = math_equal(extracted_answer, solution)
    if extracted_answer is None:
        raise ValueError('extracted answer is None')
    if is_correct:
        right_count += 1
    num_generated_tokens += tree.num_generated_tokens
    print("--------------------------------")
    print(f'question: {question}')
    print(f'solution: {solution}')
    print(f'extracted answer: {extracted_answer}')
    print(f'is correct: {is_correct}')
    print(f'Accuracy: {right_count/(i+1)}')
    wandb.log({"Accuracy": right_count/(i+1)}, step=i)
    print(f'Number of generated tokens: {tree.num_generated_tokens}')
    # print out the error steps count
    error_steps_count = 0
    for path in tree.explored_paths:
        if len(path.steps) <= 1:
            error_steps_count += 1
    error_steps_count_all += error_steps_count
    print(f'Error steps count: {error_steps_count_all}')
    wandb.log({"Error steps count": error_steps_count_all}, step=i)
    print("--------------------------------")


wandb.log({"Accuracy": right_count/len(dataset)})
wandb.log({"Number of generated tokens": num_generated_tokens/len(dataset)})
end_time = time.time()
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()

length of the dataset: 499
INFO 04-19 18:26:39 chat_utils.py:333] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it, est. speed input: 189.19 toks/s, output: 258.06 toks/s]
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


first_score_index: 0
scores: [0.8984375  0.94140625 0.96875    0.99609375 1.        ]
path.steps: ['## Step 1: Recall the conversion formulas from rectangular coordinates to polar coordinates.\nTo convert from rectangular coordinates $(x, y)$ to polar coordinates $(r, \\theta)$, we use the following formulas:\n$r = \\sqrt{x^2 + y^2}$ and $\\theta = \\tan^{-1}\\left(\\frac{y}{x}\\right)$.', '## Step 2: Apply the conversion formulas to the given point $(0,3)$.\nSubstituting $x = 0$ and $y = 3$ into the formulas, we get:\n$r = \\sqrt{0^2 + 3^2} = \\sqrt{9} = 3$\n$\\theta = \\tan^{-1}\\left(\\frac{3}{0}\\right)$', '## Step 3: Handle the case where the denominator becomes zero.\nSince $\\tan^{-1}\\left(\\frac{3}{0}\\right)$ is undefined, we must find the principal value of the angle. This occurs when $x = 0$ and $y > 0$. In this case, the angle is $\\frac{\\pi}{2}$.', '## Step 4: Write the polar coordinates.\nTherefore, the polar coordinates of the point $(0, 3)$ is $\\left(3, \\frac{\\pi}{

Processed prompts: 100%|██████████| 4/4 [00:04<00:00,  1.03s/it, est. speed input: 210.10 toks/s, output: 634.42 toks/s]


found solutions: ['\\left(3,\\frac{\\pi}{2}\\right)', '\\left(3, \\frac{3\\pi}{4}\\right)', '\\left(3, \\frac{\\pi}{3}\\right)', '\\left(3, \\frac{3\\pi}{4}\\right)']
rewards: [0.4183053970336914, 4.079050351146887e-05, 1.1620935427281154e-05, 0.0003621762008378937]
--------------------------------
question: Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$
solution: \left(3,\frac{\pi}{2}\right)
extracted answer: \left( 3, \frac{\pi}{2} \right)
is correct: True
Accuracy: 1.0
Number of generated tokens: 2917
Error steps count: 0
--------------------------------


...

Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.93s/it, est. speed input: 32.93 toks/s, output: 295.78 toks/s]


first_score_index: 3
scores: [0.99609375 0.99609375 0.984375   0.00592041 0.11474609 0.43164062
 0.4140625  0.16210938]
path.steps: ['## Step 1: We are given a recursive sequence defined by $a_{i + 1} = \\frac{1}{1 - a_i}$ for $i \\ge 1.$ We are also given that $a_3 = a_1.$\nWe need to find $(a_9)^9$.', '## Step 2: To start, we can express $a_2$ in terms of $a_1$ using the recursive formula.\n\\[a_2 = \\frac{1}{1 - a_1}\\]', '## Step 3: We can then use the recursive formula again to express $a_3$ in terms of $a_2$ and $a_1$.\n\\[a_3 = \\frac{1}{1 - a_2} = \\frac{1}{1 - \\frac{1}{1 - a_1}} = \\frac{1}{\\frac{1 - a_1 - 1}{1 - a_1}} = \\frac{1 - a_1}{-a_1} = \\frac{a_1 - 1}{a_1}\\]', '## Step 4: Since $a_3 = a_1$, we can set the expressions for $a_2$ and $a_3$ equal to each other and solve for $a_1$.\n\\[\\frac{a_1 - 1}{a_1} = a_1\\]\nSimplifying, we get\n\\[-1 = a_1^2\\]\nSo, $a_1 = -1$.', '## Step 5: Now that we have the value of $a_1$, we can use the recursive formula to find $a_9$.\nW

Processed prompts: 100%|██████████| 4/4 [00:07<00:00,  1.97s/it, est. speed input: 237.05 toks/s, output: 681.13 toks/s]


found solutions: [None, '1', '\\frac{1}{512}', '\\frac{1298 + 582\\sqrt{5}}{8}']
rewards: [8.950374241924055e-07, 0.00016800425867859487, 0.00019880365404571795, 3.3316334277096525e-08]
--------------------------------
question: A sequence $(a_n)$ is defined as follows:
\[a_{i + 1} = \frac{1}{1 - a_i}\]for $i \ge 1.$  If $a_3 = a_1,$ compute $(a_9)^9.$
solution: \frac{1}{512}
extracted answer: -1
is correct: False
Accuracy: 0.39879759519038077
Number of generated tokens: 7404
Error steps count: 0
--------------------------------
Time taken: 2748.246670484543 seconds


Accuracy,█▅▅▂▂▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Error steps count,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Number of generated tokens,▁
Accuracy,0.3988
Error steps count,0
Number of generated tokens,2733.39679
